# Day 018：MiniMind 的 MoE 分支

本 Notebook 对照 `minimind/model/model_minimind.py`，从普通 FeedForward 回顾开始，依次观察 MoE 的 router、softmax、top-k、专家分发、输出合并和 `aux_loss`。参数量与激活参数量留到 Day 019。

In [ ]:
import sys
from pathlib import Path

candidate_roots = [Path.cwd(), *Path.cwd().parents]
repo_root = next(root for root in candidate_roots if (root / 'minimind' / 'model' / 'model_minimind.py').exists())
if str(repo_root / 'minimind') not in sys.path:
    sys.path.insert(0, str(repo_root / 'minimind'))

import torch
import torch.nn.functional as F
from torch import nn
from model.model_minimind import (
    MiniMindConfig, MiniMindBlock, MiniMindForCausalLM,
    MOEFeedForward, FeedForward
)

torch.manual_seed(0)
print('repo:', repo_root)

## 1. 普通 FFN 与 MoE 的结构分支

普通 Block 的所有 token 共用一套 `FeedForward`；MoE Block 在同一位置创建多套独立的 `FeedForward`，之后由 router 决定每个 token 走哪些专家。`gate_proj` 是 FFN 内部投影，`gate` 才是 MoE router。

In [ ]:
for use_moe in (False, True):
    config = MiniMindConfig(
        hidden_size=16, num_hidden_layers=1,
        num_attention_heads=4, num_key_value_heads=2,
        use_moe=use_moe, num_experts=4,
        num_experts_per_tok=1, moe_intermediate_size=32
    )
    block = MiniMindBlock(0, config)
    print('use_moe =', use_moe)
    print('mlp type =', type(block.mlp).__name__)
    if isinstance(block.mlp, MOEFeedForward):
        print('expert count =', len(block.mlp.experts))
        print('router gate shape =', tuple(block.mlp.gate.weight.shape))
        print('expert[0] gate_proj shape =', tuple(block.mlp.experts[0].gate_proj.weight.shape))
    print()

## 2. 展平 token：`[batch, seq, hidden]` -> `[tokens, hidden]`

MoE 对每个 token 独立路由，所以把 batch 和序列两个维度合并成 token 行。这个操作不混合 token，也不拆散单个 token 的特征。

In [ ]:
x = torch.randn(3, 5, 8)
x_flat = x.view(-1, x.shape[-1])
print('x shape:', tuple(x.shape))
print('x_flat shape:', tuple(x_flat.shape))
print('one token preserved:', torch.equal(x[1, 2], x_flat[1 * 5 + 2]))

## 3. Router logits 与 softmax 概率

router 是一个 `hidden_size -> num_experts` 的无偏置 Linear。它先给每个 token 产生专家 logits，再沿专家维度做 softmax；每个 token 的概率行独立归一化为 1。

In [ ]:
config = MiniMindConfig(hidden_size=4, num_experts=3, num_experts_per_tok=1)
router = nn.Linear(config.hidden_size, config.num_experts, bias=False)
x_flat = torch.randn(2, 4)
logits = router(x_flat)
scores = F.softmax(logits, dim=-1)
print('logits shape:', tuple(logits.shape))
print('scores shape:', tuple(scores.shape))
print('scores:\n', scores)
print('row sums:', scores.sum(dim=-1))

## 4. top-k 选择与权重归一化

`topk_idx` 是专家编号，`topk_weight` 是对应概率。`norm_topk_prob` 在 `k>1` 时把保留的权重重新归一化；它不改变选中的专家集合。

In [ ]:
scores = torch.tensor([[0.4742, 0.2161, 0.3096],
                       [0.2583, 0.4638, 0.2779]])
for k in (1, 2):
    topk_weight, topk_idx = torch.topk(scores, k=k, dim=-1, sorted=False)
    normalized = topk_weight / topk_weight.sum(dim=-1, keepdim=True)
    print(f'k={k}')
    print('indices:', topk_idx)
    print('weights:', topk_weight)
    print('normalized weights:', normalized)

## 5. `mask`、`token_idx` 与 `index_add_`

每个专家只取属于自己的 token 行。专家输出乘路由权重，再按原 token 行号加回 `y`。当 `k>1` 时，同一行会被多个专家多次累加。

In [ ]:
topk_idx = torch.tensor([[0], [2], [0], [1]])
for expert_id in range(3):
    mask = topk_idx == expert_id
    token_idx = mask.any(dim=-1).nonzero().flatten()
    print(f'expert {expert_id} -> token_idx {token_idx.tolist()}')

y = torch.zeros(4, 3)
token_idx = torch.tensor([0, 2])
weighted_output = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
y.index_add_(0, token_idx, weighted_output)
print('y after index_add_:\n', y)

## 6. 直接运行真实 `MOEFeedForward`

下面的前向传播完整执行展平、router、top-k、专家分发和恢复形状。训练模式会计算 `aux_loss`；评估模式把它置为 0。

In [ ]:
config = MiniMindConfig(
    hidden_size=4, num_experts=3, num_experts_per_tok=1,
    moe_intermediate_size=8, router_aux_loss_coef=5e-4
)
moe = MOEFeedForward(config)
x = torch.randn(1, 2, 4)

moe.train()
y_train = moe(x)
print('train output shape:', tuple(y_train.shape))
print('train aux_loss:', moe.aux_loss.item())

moe.eval()
y_eval = moe(x)
print('eval output shape:', tuple(y_eval.shape))
print('eval aux_loss:', moe.aux_loss.item())

## 7. `aux_loss` 的两个组成部分

`load` 统计实际选择结果；`scores.mean(0)` 统计 router 的平均概率偏好。极端集中路由会比均衡路由产生更大的辅助损失。

In [ ]:
def aux_from_parts(load, mean_scores, num_experts=4, coef=5e-4):
    load = torch.tensor(load, dtype=torch.float32)
    mean_scores = torch.tensor(mean_scores, dtype=torch.float32)
    dot = (load * mean_scores).sum()
    return dot.item(), (dot * num_experts * coef).item()

cases = {
    '集中': ([1, 0, 0, 0], [0.7, 0.1, 0.1, 0.1]),
    '部分集中': ([0.5, 0.5, 0, 0], [0.6, 0.3, 0.05, 0.05]),
    '均衡': ([0.25] * 4, [0.25] * 4),
}
for name, (load, mean_scores) in cases.items():
    print(name, '-> dot, aux_loss =', aux_from_parts(load, mean_scores))

## 8. 多层 MoE 的 `aux_loss` 汇总

每一层有自己的 router 和专家。`MiniMindModel` 把所有 MoE 层的辅助损失相加，再由 `MiniMindForCausalLM` 返回；训练循环使用 `res.loss + res.aux_loss` 进行反向传播。

In [ ]:
config = MiniMindConfig(
    vocab_size=20, hidden_size=8, num_hidden_layers=2,
    num_attention_heads=2, num_key_value_heads=1,
    use_moe=True, num_experts=3, num_experts_per_tok=1,
    moe_intermediate_size=16, max_position_embeddings=32, flash_attn=False
)
model = MiniMindForCausalLM(config)
model.train()
outputs = model(torch.tensor([[1, 4, 7, 2]]))
layer_losses = [layer.mlp.aux_loss for layer in model.model.layers if isinstance(layer.mlp, MOEFeedForward)]
print('layer aux_losses:', [x.item() for x in layer_losses])
print('model aux_loss:', outputs.aux_loss.item())
print('manual sum:', sum(x.item() for x in layer_losses))
assert torch.isclose(outputs.aux_loss, sum(layer_losses))

## 今日边界

已验证：普通 FFN 与 MoE 的结构差异、逐 token 路由、softmax、top-k、专家分发、加权合并、`aux_loss` 以及多层汇总。

Day 019 再继续统计总参数量、激活参数量，并对照实际专家调用和计算成本。